# EPIC Clarity Death Hydration

This notebook hydrates the OMOP DEATH table from EPIC Clarity patient death dates.

## Source Tables
- `_exponent._bronze_epic_clarity_*.dbo_PATIENT` - Primary death_date field
- `_exponent._bronze_epic_clarity_*.dbo_PATIENT_4` - External/supplemental death_date source

## OMOP Fields Populated
- person_id
- death_date (coalesce internal and external sources)
- death_type_concept_id

In [ ]:
source = 'epic_clarity'

In [ ]:
silver_death_df = spark.sql("""
SELECT
    m.person_id,
    p.DEATH_DATE AS death_date,
    'epic_clarity' AS source_system,
    CURRENT_TIMESTAMP() AS last_mod_tsp
FROM _exponent._bronze_epic_clarity.patient p
INNER JOIN _exponent.omop_mapping.source_to_person m
    ON CONCAT_WS(CHR(31), 'epic_clarity', 'PATIENT', 'PAT_ID', CAST(p.PAT_ID AS STRING)) = m.person_source_value
    AND m.active_flag = TRUE
WHERE p.DEATH_DATE IS NOT NULL
""")

display(silver_death_df)
silver_death_df.createOrReplaceTempView("silver_death")

In [ ]:
%sql
MERGE INTO _exponent.omop_silver.death AS target
USING silver_death AS source
ON target.person_id = source.person_id

WHEN MATCHED AND NOT (
    target.death_date <=> source.death_date
)
THEN UPDATE SET
    target.death_date = source.death_date,
    target.last_mod_tsp = source.last_mod_tsp

WHEN NOT MATCHED THEN INSERT (
    person_id,
    death_date,
    source_system,
    last_mod_tsp
)
VALUES (
    source.person_id,
    source.death_date,
    source.source_system,
    source.last_mod_tsp
)

In [ ]:
gold_death_df = spark.sql("""
SELECT
    s.person_id,
    s.death_date,
    0 AS death_type_concept_id
FROM _exponent.omop_silver.death s
WHERE s.source_system = 'epic_clarity'
""")

display(gold_death_df)
gold_death_df.createOrReplaceTempView("gold_death")

In [ ]:
%sql
MERGE INTO _exponent.omop.death AS target
USING gold_death AS source
ON target.person_id = source.person_id

WHEN MATCHED AND NOT (
    target.death_date <=> source.death_date
)
THEN UPDATE SET
    target.death_date = source.death_date,
    target.death_type_concept_id = source.death_type_concept_id

WHEN NOT MATCHED THEN INSERT (
    person_id,
    death_date,
    death_type_concept_id
)
VALUES (
    source.person_id,
    source.death_date,
    source.death_type_concept_id
)